In [ ]:






# Kaggle Competition Submission: Structured News Summarizer with Gemini 2.0

## Project Overview
##This notebook demonstrates a Generative AI pipeline using Gemini 2.0 from Google.
##The project showcases structured summarization and evaluation capabilities through few-shot prompting and auto-rating of responses. 
##The model answers user queries with clear structure and then evaluates its own response.


## 1. Setup Environment


import io
import enum
from IPython.display import Markdown, display
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient


# Load API Key from Kaggle secrets
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)


## 2. Define Prompts and Evaluation Rubric

# Few-shot prompt format
FEW_SHOT_PROMPT = """You are a helpful assistant that provides structured answers to user questions. Use clear, concise, and grounded language.

## Example 1
Q: What is the greenhouse effect?
A:
**Summary:**
The greenhouse effect is the process by which gases in Earth’s atmosphere trap heat, keeping the planet warm enough to sustain life.

**Key Points:**
- Greenhouse gases like CO₂ and methane trap heat from the sun.
- This effect is natural and necessary, but too much can lead to global warming.
- Human activity has increased greenhouse gas levels.

**Sources:**
[1] NASA Climate
[2] EPA - Greenhouse Gas Basics

## Example 2
Q: What caused the 2008 financial crisis?
A:
**Summary:**
The 2008 financial crisis was triggered by the collapse of the housing market and risky banking practices in the U.S.

**Key Points:**
- Subprime mortgage defaults led to banking losses.
- Lehman Brothers collapsed, triggering panic.
- Government bailouts followed to stabilize markets.

**Sources:**
[1] Federal Reserve History
[2] Investopedia - 2008 Crash Explained

## Now answer this:
Q: {question}
A:
"""


# Evaluation prompt
EVAL_PROMPT = """
# Instruction
You are an expert evaluator. Your task is to rate the AI's structured answer on:

1. Instruction following
2. Groundedness
3. Conciseness
4. Fluency

Use the rubric:
5 = Very Good
4 = Good
3 = Okay
2 = Poor
1 = Very Bad

# Prompt
{prompt}

# AI Response
{response}
"""

# Rating enum
class Rating(enum.Enum):
    VERY_GOOD = '5'
    GOOD = '4'
    OK = '3'
    BAD = '2'
    VERY_BAD = '1'

## 3. Generate and Evaluate Structured Answers

# Structured Answer Generator with Evaluation
model = genai.GenerativeModel("gemini-2.0-flash") 

def generate_structured_answer(question, retries=2):
    best_response = None
    best_score = 0

    for _ in range(retries):
        prompt = FEW_SHOT_PROMPT.format(question=question)
        try:
            response = model.generate_content(prompt)
            text = response.text

            rating_prompt = EVAL_PROMPT.format(prompt=question, response=text)
            rating_response = model.generate_content(rating_prompt)
            score = next((int(s) for s in rating_response.text.split() if s.isdigit()), 0)

            if score > best_score:
                best_score = score
                best_response = text

            if best_score == 5:
                break

        except Exception as e:
            print("Error generating or rating response:", e)

    return best_response, best_score

def display_response(response, score):
    display(Markdown(f"**Gemini Answer (rated {score}/5):**\n\n{response}"))

## 4. Interactive Q&A

while True:
    user_input = input("Ask a question (or type 'exit'): ")
    if user_input.lower() == "exit":
        print("Goodbye!")
        break

    try:
        answer, score = generate_structured_answer(user_input)
        display_response(answer, score)
    except Exception as e:
        print("Error:", str(e))

## Final Notes for Reviewers
##- The model used is Gemini 2.0 (via `google-genai==1.7.0`)
##- All answers are structured using few-shot prompting.
##- Evaluation is auto-generated using a rubric-based prompt.
##- This is a **work-in-progress** version of the original project goal: *A news summarizer with grounding and evaluation.*
##- Some grounding and citation features are yet to be fully integrated.

##Thank you for reviewing!
